# L83 · Transformer 从零复现 — 多头注意力 + 位置编码 + Feed-Forward 完整实现

**目标**：从头实现 `scaled_dot_product_attention`，理解 Q/K/V 的几何含义，掌握缩放因子 `√d_k` 的必要性和 multi-head 的作用。

🔗 **Aurora 连接**：本节实现的注意力核心是 `aurora/whisper/` 编码器和 Month 5 本地 LLM 解码器的共同基础模块；后续 Whisper 编码器将直接调用这里导出的 `scaled_dot_product_attention`。

Transformer 的核心直觉：每个位置的输出是对所有位置 **值（V）** 的加权求和，权重由当前位置的**查询（Q）**与所有位置的**键（K）**的相似度决定。缩放点积让相似度在高维空间里不会饱和，softmax 把相似度归一化成概率分布。这一机制天然支持并行计算，是取代 RNN 的关键。

In [ ]:
import numpy as np

## 1. Attention(Q, K, V) = softmax(QKᵀ / √d_k) · V

设序列长度为 `seq`，特征维度为 `d_k`（键/查询维）和 `d_v`（值维）。

- **Q**（Query）：当前位置想要「查什么」
- **K**（Key）：每个位置提供的「索引标签」
- **V**（Value）：每个位置实际携带的信息

计算步骤：
```
scores  = Q @ Kᵀ          # (seq_q, seq_k)  点积相似度
scores /= sqrt(d_k)        # 缩放
weights = softmax(scores)  # 每行归一化为概率
output  = weights @ V      # (seq_q, d_v)  加权聚合
```

几何上，`Q[i] · K[j]` 是向量夹角的余弦乘以模长——Q 和 K 方向越接近，该位置权重越高。

In [ ]:
# 演示：2个查询，3个键/值，d_k=4，d_v=6
np.random.seed(0)
Q_demo = np.random.randn(2, 4)   # (seq_q=2, d_k=4)
K_demo = np.random.randn(3, 4)   # (seq_k=3, d_k=4)
V_demo = np.random.randn(3, 6)   # (seq_k=3, d_v=6)

scores_demo = Q_demo @ K_demo.T  # (2, 3)
print('scores shape:', scores_demo.shape)
print('scores (未缩放):\n', scores_demo.round(3))

## 2. 为什么除以 √d_k？

设 Q、K 的分量是均值0方差1的独立随机变量，则点积 `Q · K = Σ Q_i K_i` 的方差为 `d_k`，标准差为 `√d_k`。

当 `d_k` 很大（例如 512），点积的绝对值会很大，进入 softmax 的饱和区（某一项接近 1，其余接近 0），梯度几乎为 0，训练停滞。

除以 `√d_k` 后方差归一化为 1，softmax 输入数量级可控：
```
Var(Q · K / √d_k) = d_k / d_k = 1
```

In [ ]:
# 演示：d_k 对 softmax 输出分布的影响
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

rng = np.random.default_rng(42)
for d_k in [4, 64, 512]:
    q = rng.standard_normal(d_k)
    k = rng.standard_normal((8, d_k))
    raw   = softmax(q @ k.T)
    scaled = softmax(q @ k.T / np.sqrt(d_k))
    print(f'd_k={d_k:4d} | 未缩放最大权重={raw.max():.3f}  缩放后最大权重={scaled.max():.3f}')

## 3. Multi-Head Attention

单头注意力只能学到一种「相关性」。Multi-head 的做法：

```
将 d_model 拆成 h 份，每份维度 d_k = d_model / h
head_i = Attention(Q·Wq_i, K·Wk_i, V·Wv_i)   # 独立的线性投影
output = Concat(head_0, ..., head_{h-1}) · Wo  # concat 再投影
```

每个头在不同子空间里做注意力，可以同时关注「语法依存」「语义相似」「位置临近」等不同模式，最后线性变换融合。

本节只实现单头核心 `scaled_dot_product_attention`；multi-head 的投影矩阵 `Wq/Wk/Wv/Wo` 将在 L2 中封装进 `MultiHeadAttention` 类。

In [ ]:
# 演示：multi-head 概念——把 d_model=16 拆成 h=2 个头
d_model, h = 16, 2
d_k = d_model // h
rng = np.random.default_rng(7)
x = rng.standard_normal((1, 5, d_model))  # (B=1, seq=5, d_model=16)

# 简化：用切片模拟投影（真实实现用 Wq @ x）
for i in range(h):
    Q_h = x[:, :, i*d_k:(i+1)*d_k]
    K_h = x[:, :, i*d_k:(i+1)*d_k]
    V_h = x[:, :, i*d_k:(i+1)*d_k]
    print(f'head {i}: Q shape {Q_h.shape}, d_k={d_k}')

## 4. ✏️ 实现 `scaled_dot_product_attention(Q, K, V, mask=None)`

**签名**：`Q, K, V: np.ndarray` shape `(B, seq_q, d_k)` / `(B, seq_k, d_k)` / `(B, seq_k, d_v)` → `np.ndarray` shape `(B, seq_q, d_v)`

**推理路线**：
1. `scores = Q @ K.transpose(0,2,1) / np.sqrt(Q.shape[-1])`  →  shape `(B, seq_q, seq_k)`
2. 如果 `mask` 不为 None，将 `mask==True` 的位置置为 `-np.inf`（softmax 后变 0）
3. `weights = softmax(scores, axis=-1)`  →  每行归一化，和为 1
4. `return weights @ V`  →  shape `(B, seq_q, d_v)`

**参考输入输出**：
```
B=2, seq_q=4, seq_k=6, d_k=8, d_v=16
Q: (2,4,8)   K: (2,6,8)   V: (2,6,16)
scores: (2,4,6)   weights: (2,4,6) 每行和=1
output: (2,4,16)
```

<details><summary>点击查看参考实现</summary>

```python
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    # (B, seq_q, seq_k)
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)
    if mask is not None:
        scores = scores.copy()
        scores[mask] = -np.inf
    # softmax along seq_k axis
    scores -= scores.max(axis=-1, keepdims=True)  # 数值稳定
    weights = np.exp(scores)
    weights /= weights.sum(axis=-1, keepdims=True)
    return weights @ V
```

</details>

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (B, seq_q, d_k)
    K: (B, seq_k, d_k)
    V: (B, seq_k, d_v)
    mask: bool array (B, seq_q, seq_k), True 表示屏蔽该位置
    返回: (B, seq_q, d_v)
    """
    d_k = Q.shape[-1]
    # ✏️ TODO: 计算缩放点积 scores，shape (B, seq_q, seq_k)
    scores = None

    # ✏️ TODO: 如果 mask 不为 None，屏蔽对应位置（置 -inf）

    # ✏️ TODO: softmax(scores, axis=-1) → weights
    weights = None

    # ✏️ TODO: 返回 weights @ V
    return None

In [ ]:
# 形状检查
np.random.seed(1)
Q = np.random.randn(2, 4, 8)
K = np.random.randn(2, 6, 8)
V = np.random.randn(2, 6, 16)

out = scaled_dot_product_attention(Q, K, V)
assert out.shape == (2, 4, 16), f'形状错误: {out.shape}'
print('✅ output shape:', out.shape)

# 权重行和检查（需要在实现中暴露 weights，或单独验证 softmax）
d_k = Q.shape[-1]
scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)
scores_s = scores - scores.max(axis=-1, keepdims=True)
w = np.exp(scores_s); w /= w.sum(axis=-1, keepdims=True)
assert np.allclose(w.sum(axis=-1), 1.0), '权重行和不为 1'
print('✅ softmax 权重每行和为 1')

# Causal mask 检查：下三角遮住未来
B, seq = 1, 4
Q2 = np.random.randn(B, seq, 8)
K2 = np.random.randn(B, seq, 8)
V2 = np.random.randn(B, seq, 8)
mask = np.triu(np.ones((B, seq, seq), dtype=bool), k=1)  # 上三角=True
out_masked = scaled_dot_product_attention(Q2, K2, V2, mask=mask)
assert out_masked.shape == (B, seq, 8)
print('✅ causal mask 输出形状正确:', out_masked.shape)

## 5. 参数实验：Q 方向对注意力权重的影响

固定 `K`（3个键）和 `V`，改变 `Q` 的方向，观察 `weights` 如何随 `Q` 与各 `K` 夹角变化。

**预期现象**：
- 将 `Q` 设为与 `K[0]` 完全平行（`Q = K[0]`）→ `weights[0]` 接近 1
- 将 `Q` 设为均匀方向（`Q` 与所有 `K` 夹角相同）→ `weights` 趋于均匀分布 `[1/3, 1/3, 1/3]`
- `d_k` 越大，未缩放时权重越极端（更尖锐）

In [ ]:
np.random.seed(3)
d_k = 8
K_exp = np.random.randn(1, 3, d_k)  # 3个键
V_exp = np.random.randn(1, 3, d_k)

# 实验1：Q 与 K[0] 平行
Q_parallel = K_exp[:, [0], :]  # shape (1,1,d_k)
out1 = scaled_dot_product_attention(Q_parallel, K_exp, V_exp)
scores1 = Q_parallel @ K_exp.transpose(0,2,1) / np.sqrt(d_k)
s1 = scores1 - scores1.max(axis=-1, keepdims=True)
w1 = np.exp(s1); w1 /= w1.sum(axis=-1, keepdims=True)
print('Q ∥ K[0] → weights:', w1[0,0].round(3))

# 实验2：Q 为零向量（均匀注意力）
Q_zero = np.zeros((1, 1, d_k))
scores2 = Q_zero @ K_exp.transpose(0,2,1) / np.sqrt(d_k)
s2 = scores2 - scores2.max(axis=-1, keepdims=True)
w2 = np.exp(s2); w2 /= w2.sum(axis=-1, keepdims=True)
print('Q = 0 (均匀)   → weights:', w2[0,0].round(3))

# 实验3：d_k 对尖锐度的影响
print('\n--- d_k 对尖锐度的影响（未缩放 vs 缩放）---')
for dk in [4, 64, 512]:
    rng2 = np.random.default_rng(99)
    q_ = rng2.standard_normal((1,1,dk))
    k_ = rng2.standard_normal((1,3,dk))
    sc_raw = q_ @ k_.transpose(0,2,1)
    sc_scl = sc_raw / np.sqrt(dk)
    def sw(s):
        s = s - s.max(axis=-1, keepdims=True)
        e = np.exp(s); return e / e.sum(axis=-1, keepdims=True)
    print(f'  d_k={dk:4d}: 未缩放max={sw(sc_raw)[0,0].max():.3f}  缩放max={sw(sc_scl)[0,0].max():.3f}')

## 本课小结

`scaled_dot_product_attention` 以 `(B, seq_q, d_k)` 的 Q/K 和 `(B, seq_k, d_v)` 的 V 为输入，输出 `(B, seq_q, d_v)` 的上下文向量——每行是对所有 V 的加权平均。
缩放因子 `1/√d_k` 保持 softmax 输入的方差恒为 1，避免梯度消失，是训练稳定的关键。
该函数将直接嵌入 `aurora/whisper/attention.py` 的 `MultiHeadAttention` 模块，作为 Whisper 编码器和 Month 5 本地 LLM 解码器的共同基础。
下节（M5-L2）将在此基础上封装投影矩阵 `Wq/Wk/Wv/Wo`，实现完整的 `MultiHeadAttention` 层。